In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

def near_psd(a, epsilon: float = 0.0):
    # keep labels
    idx, cols = a.index, a.columns
    A = a.to_numpy(float)
    A = 0.5 * (A + A.T)

    d = np.diag(A).copy()
    d[d < 0] = 0.0
    np.fill_diagonal(A, d)

    # to correlation if needed
    diagA = np.diag(A)
    is_corr = np.allclose(diagA, np.ones_like(diagA), atol=1e-12)
    if not is_corr:
        std = np.sqrt(np.maximum(diagA, 0.0))
        std = np.where(std < 1e-12, 1e-12, std)
        R = (A / std[:, None]) / std[None, :]
    else:
        R = A

    # eigen clip + Higham-like scaling
    vals, vecs = np.linalg.eigh(R)
    vals = np.maximum(vals, epsilon)
    denom = (vecs**2) @ vals
    denom = np.where(denom < 1e-18, 1e-18, denom)
    T = np.diag(np.sqrt(1.0 / denom))
    L = np.diag(np.sqrt(vals))
    B = T @ vecs @ L
    R_psd = B @ B.T
    np.fill_diagonal(R_psd, 1.0)
    R_psd = 0.5 * (R_psd + R_psd.T)

    # back to covariance if needed
    if not is_corr:
        C_psd = (R_psd * std[:, None]) * std[None, :]
    else:
        C_psd = R_psd

    C_psd = 0.5 * (C_psd + C_psd.T)
    return pd.DataFrame(C_psd, index=idx, columns=cols)

# Read data and give output
DATA_DIR = Path.cwd() / "testfiles_" / "data"
csv_path = DATA_DIR / "testout_1.3.csv"

df = pd.read_csv(csv_path, header=0)
# force numeric
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

nrows, ncols = df.shape
if nrows != ncols:
    raise ValueError(f"Matrix must be square, got {nrows}x{ncols}")

# set row labels = column labels because the file has no index column
df.index = df.columns

near_cov = near_psd(df)
print(near_cov)

          x1        x2        x3        x4        x5
x1  1.173986 -0.617989 -0.284559 -0.065152 -0.688287
x2 -0.617989  1.318197  0.017092  0.445696  0.139176
x3 -0.284559  0.017092  0.918102  0.354147  0.246056
x4 -0.065152  0.445696  0.354147  0.894764 -0.218717
x5 -0.688287  0.139176  0.246056 -0.218717  0.522607
